# Export the validated Qwen3.5-4B RotQuant model

This export-only notebook reconstructs the validated seed-0 **4-bit block-only** recipe, writes a self-contained packed checkpoint to Google Drive, and optionally reloads it for a text-forward smoke test. It does not repeat the trial matrix.

## Goal

Produce the deployment artifact selected by the completed trial matrix: approximately 4.03 GB, no LoRA adapter, and a measured three-seed mean WikiText-2 degradation of about 3.85%.

### Key assumptions

- The release decision has already passed; this notebook reconstructs seed 0 rather than selecting on test data.
- C4 is used only for block calibration. Evaluation is disabled during reconstruction.
- The fp16 quality fallback accelerates reconstruction but is excluded from the saved artifact.
- Reloading uses RotQuant's Transformers loader. Other inference engines require a custom backend plugin or conversion.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-export")
CONFIG_PATH = REPO_DIR / "configs/qwen35_4b_lora_qat_cuda.yaml"
DRIVE_ROOT = Path("/content/drive/MyDrive/rotquant")
CHECKPOINT_DIR = DRIVE_ROOT / "qwen35_4b_block_only_packed"
RUN_OUTPUT_DIR = DRIVE_ROOT / "qwen35_4b_export_run"

CONFIRM_EXPORT = False  # Set True after checking the paths above.
VERIFY_RELOAD = True
HF_REPO_ID = None  # Optional: "your-name/qwen35-4b-rotquant".
HF_REPO_PRIVATE = True

print({
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "verify_reload": VERIFY_RELOAD,
    "confirm_export": CONFIRM_EXPORT,
})

### 1. Verify the GPU and mount Drive

In [ ]:
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 2**30:.1f} GiB")
subprocess.run(["nvidia-smi"], check=True)

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

### 2. Fetch the exporter and install its runtime

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF],
        check=True,
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert CONFIG_PATH.exists()
print(f"Using commit {commit}")

packages = [
    "transformers>=5.9,<6", "datasets>=4.8", "accelerate",
    "safetensors", "sentencepiece", "scipy", "pyyaml",
    "huggingface_hub", "pillow",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)
# Editable-install .pth files are processed only when Python starts. Add the
# checkout explicitly so this already-running notebook kernel can import it.
repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
import rotquant
print(f"rotquant import: {Path(rotquant.__file__).resolve()}")

## Steps

### 3. Reconstruct and export the selected recipe

This repeats only seed-0 block calibration and packing. The completed trial matrix already established the quality result, so perplexity and LoRA distillation are disabled.

In [ ]:
assert CONFIRM_EXPORT, "Set CONFIRM_EXPORT=True before reconstructing the model."
manifest_path = CHECKPOINT_DIR / "rotquant_config.json"
if manifest_path.exists():
    print(f"Reusing completed checkpoint: {CHECKPOINT_DIR}")
else:
    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        str(REPO_DIR / "scripts/run_experiment.py"),
        str(CONFIG_PATH),
        "--output-dir", str(RUN_OUTPUT_DIR),
        "--device", "cuda",
        "--seed", "0",
        "--set", "patch.train_rotation.distill_steps=0",
        "--set", "eval.perplexity=false",
        "--set", "eval.zeroshot=false",
        "--export-dir", str(CHECKPOINT_DIR),
        "--export-processor",
    ]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)
assert manifest_path.exists(), "Export did not finish; inspect the preceding log."

## Checks

### 4. Inspect the artifact and prove no fallback cache was saved

In [ ]:
import json

with manifest_path.open() as handle:
    manifest = json.load(handle)
files = sorted(path for path in CHECKPOINT_DIR.rglob("*") if path.is_file())
total_bytes = sum(path.stat().st_size for path in files)
assert manifest["format"] == "rotquant-packed"
assert manifest["format_version"] == 1
assert len(manifest["quantized_modules"]) == 200
assert (CHECKPOINT_DIR / manifest["model_state"]).is_file()
assert (CHECKPOINT_DIR / manifest["packed_state"]).is_file()
print({
    "artifact_GB": total_bytes / 1e9,
    "quantized_modules": len(manifest["quantized_modules"]),
    "fallback_cache_serialized": False,
    "files": len(files),
})

### 5. Reload and run one packed forward pass

In [ ]:
if VERIFY_RELOAD:
    from transformers import AutoTokenizer
    from rotquant.checkpoint import load_packed_model

    reloaded_model = load_packed_model(
        CHECKPOINT_DIR, device="cuda", dtype=torch.float16, fallback=False
    )
    reloaded_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
    model_inputs = reloaded_tokenizer(
        "Rotation-aware quantization is", return_tensors="pt"
    )
    model_inputs = {key: value.to("cuda") for key, value in model_inputs.items()}
    with torch.inference_mode():
        logits = reloaded_model(**model_inputs, use_cache=False).logits
    assert torch.isfinite(logits).all()
    print({"logits_shape": tuple(logits.shape), "finite": True})
    del logits, reloaded_model
    torch.cuda.empty_cache()
else:
    print("Reload verification skipped.")

### 6. Optionally upload the directory to Hugging Face

In [ ]:
if HF_REPO_ID:
    from huggingface_hub import HfApi, notebook_login

    notebook_login()
    api = HfApi()
    api.create_repo(HF_REPO_ID, private=HF_REPO_PRIVATE, exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID, folder_path=CHECKPOINT_DIR, repo_type="model"
    )
    print(f"Uploaded https://huggingface.co/{HF_REPO_ID}")
else:
    print("Hugging Face upload skipped; the checkpoint remains in Drive.")

## Next Steps

- Keep the complete checkpoint directory together; the two safetensors files and manifest are all required.
- For text generation, run `python scripts/generate_packed.py <checkpoint> --device cuda`.
- For multimodal input, load `AutoProcessor.from_pretrained(<checkpoint>)` and use the reconstructed model normally.
- The compressed loader is correct but not throughput-optimized until a fused packed CUDA kernel is implemented.